# Does behaviour match the declared profile? (AML transaction monitoring)

**Eskwelabs Data Analytics Bootcamp · Sprint 2 · team project**

This is an extract of my team's notebook, *Improve Investigation Efficiency While Reducing False
Positive Workload*. It keeps only the parts I wrote:

- the **Executive Summary** and **Business Context**, and
- **Investigation 4**, which compares the money each customer moved in 2025 with the income they
  declared when they opened their account.

The other investigations were written by my teammates and are not included.

> **Public copy.** Outputs are cleared and no data is included. The bootcamp's BigQuery project
> and dataset names are replaced with placeholders (`your-gcp-project.aml_dataset`). The dataset
> is a synthetic AML scenario provided by the bootcamp.

## Setup

Connects to BigQuery. `alerts_df` (one row per alert) is built by the team's preparation query,
included below because step 4 of Investigation 4 needs it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

from google.colab import auth
from google.cloud import bigquery

PROJECT_ID = "your-gcp-project"   # placeholder
auth.authenticate_user(project_id=PROJECT_ID)
client = bigquery.Client(project=PROJECT_ID)

# Executive Summary

## The problem

The bank watches every transaction for signs of money laundering. When something looks unusual, the system raises an **alert** — a flag asking a person to take a look.

In 2025 the system raised 2,618 alerts. The team of 24 investigators worked through 1,623 of them.

**The other 995 were never opened by anyone.**

Our client, the Head of AML Transaction Monitoring, cannot fix this by hiring. So the one thing she can change is the **order** the alerts get looked at — and right now nothing decides that order except chance.

## The question we set out to answer

> Which alerts, if looked at first, would find more of the activity worth reporting — using the same number of hours the team already has?

We are not trying to catch criminals. The data does not say who is one, and no combination of the information we have would tell us. We are trying to build a better queue.

## What we found

**1. The biggest rule creates the most work and finds the least.**
One rule flags any single payment of PHP 500,000 or more. It raises 1,540 of the 2,618 alerts — nearly six in ten. Of the ones investigators worked, **seven in ten were closed with nothing found.** In fairness, this rule exists because the law requires the bank to report payments that size, so it is doing its job. But it is where most of the team's time goes.

**2. The smallest rule finds the most.**
A rule that spots money arriving and leaving again quickly raises only 124 alerts — five in a hundred. But **three out of four of those led to a report or a referral**, against an average of one in four across all alerts. The rule making the least noise is the most useful one the bank has.

**3. Payments cluster just under the PHP 500,000 reporting line.**
865 payments landed between PHP 450,000 and PHP 499,999 — just below the amount that must be reported. Ninety-one customers made 794 of them. One customer did it nineteen times. Splitting a large sum into smaller pieces to stay under a reporting limit is a known tactic, but ordinary payments can look identical, so this is a reason to look, not a conclusion.

**4. The customers whose spending least matches their stated income are students and the unemployed.**
When customers open an account they say what they do and what they earn. A typical student moved 7.7% of their stated income during the year. A typical unemployed customer, 6.0%. A typical business owner, 0.4%. That is the opposite of what most people expect — and it matters, because people recruited to move money for someone else are usually students or out of work. Stated income was never checked by anyone, so this points at accounts worth reviewing, not at people.

**5. The rule for sleeping accounts scores well and finds least.**
When a long-inactive account suddenly becomes busy, a rule flags it. It raised 126 alerts and 83 were worked. Only 15 led to anything — the weakest result of the five rules.

## What we recommend

| What to do | How long | What it costs |
| ---------- | -------- | ------------- |
| Look at alerts from the two best-performing rules before the big-payment ones | Next month | Nothing. Same work, different order |
| Write down a reason whenever an alert is left unopened | Now | Seconds per alert. Closes a gap no one can currently explain |
| Review repeat big-payment alerts on customers already cleared before, in batches | Three months | Process change only |
| Look for repeated near-limit payments by the same customer, not just single ones | Six months | Some development work |

## What we cannot say

Nothing in this data identifies a money launderer. There is no such field, and none can be worked out from what we have. Everything above describes behaviour **worth a closer look**, not proof that anyone did anything wrong.

We only know the outcome of the 1,623 alerts someone chose to open. Those judgements were made by people under time pressure, using the same rules we are assessing. So none of it tells us what was in the 995 nobody opened.

Finally, about seven in ten payments go to accounts outside this bank, so we usually cannot see who received the money. Everything here is based on the sending side.

# **BUSINESS CONTEXT**

**The question:** Why does the way alerts are handled today need to change?

## **Our client**

She runs the team that watches the bank's transactions for signs of money laundering, at a mid-sized Philippine bank. Her problem is not that she doesn't know what to look for. It is that there is more to look at than there are people to look.

In 2025, five automatic rules raised 2,618 alerts. Her 24 investigators opened 1,623 of them — about 68 each. The remaining 995 sat in a queue that never emptied.

She cannot hire more people. So the only thing she can change is which alerts get looked at first. And here is the awkward part: **somebody decided which 1,623 to open, and that decision was never written down.** It cannot be checked, and it cannot be improved.

## **How the process works, with real numbers**

| Stage | What happens | How many |
| ----- | ------------ | -------- |
| Customers | People who bank here | 10,000 |
| Accounts | Accounts they hold | 12,000 |
| Payments | Money moved during 2025 | 46,793 |
| Alerts | Automatic rules flag something unusual | 2,618 |
| Cases opened | An investigator actually looks | 1,623 |
| Something found | Reported to the regulator, or referred internally | 397 |

Nearly 47,000 payments narrow down to 397 outcomes. **The part in the middle — the 2,618 flags and the 1,623 cases — is where every hour of the team's time goes.**

*(We use 12,000 accounts and 46,793 payments rather than the raw file counts, because a handful of records appear twice in the source. The Data Validation section shows how we checked this.)*

##**The Monitoring Pipeline**

-Each stage below feeds the next, and volume falls sharply between them.

| Stage | What happens | Volume |
| ----- | ------------ | ------ |
| Customers onboarded | KYC profile captured, risk score assigned | 10,000 |
| Accounts opened | Accounts linked to owning customers | 12,028 |
| Transactions monitored | Every monetary movement during 2025 | 46,838 |
| Alerts raised | Five rules fire on individual transactions | 2,618 |
| Alerts investigated | An investigator opens and works a case | 1,623 |
| Filed report or escalation | Case ends in SAR_Filed or Escalated | 397 |

-995 alerts, 38% of all alerts raised, were never worked.

-Accounts and transactions contain duplicate identifiers (28 and 45 respectively); distinct counts are 12,000 and 46,793. See the Data Validation section for the reconciliation.

**This table is our process diagram.** A flowchart drawn by hand would show the same six boxes with no numbers in them. Presenting the pipeline as a funnel with real volumes carries strictly more information, and it stays inside the techniques this notebook uses.

In [ ]:
# The funnel proper: four stages in one chain, each a subset of the stage above it.
# Written narrowest-first so plt.barh draws the widest stage at the top.
funnel_data = pd.DataFrame({
    "stage": [
        "Filed report or escalated",
        "Alerts investigated",
        "Alerts raised",
        "Transactions monitored"
    ],
    "volume": [397, 1623, 2618, 46838]
})

# Customers (10,000) and accounts (12,028) are deliberately not on this chart.
# They are the standing population, not stages a transaction passes through,
# and they are counted in a different unit, so they do not share this axis.

# Set figure size
plt.figure(figsize = (8, 3.5), dpi = 200)

# Run horizontal bar plot
plt.barh(
    funnel_data["stage"],
    funnel_data["volume"]
)

# Set title
plt.title("Monitoring Funnel, 2025: Transactions to Action")

# Set labels
plt.xlabel("Number of records")
plt.ylabel("Monitoring stage")

# Show figure
plt.show()

# Second view: the alert queue alone, where investigator time is actually spent
alert_funnel = pd.DataFrame({
    "stage": ["Filed report or escalated", "Alerts investigated", "Alerts raised"],
    "volume": [397, 1623, 2618]
})

# Set figure size
plt.figure(figsize = (8, 2.5), dpi = 200)

# Run horizontal bar plot
plt.barh(
    alert_funnel["stage"],
    alert_funnel["volume"]
)

# Set title
plt.title("Alert Queue, 2025")

# Set labels
plt.xlabel("Number of alerts")
plt.ylabel("Stage")

# Show figure
plt.show()

-Out of 46,838 transactions, 2,618 raised an alert and fewer than 400 led to any action. The funnel narrows sharply, and the middle of it — the 2,618 alerts and the 1,623 cases opened against them — is where her team's time goes.

## **The five rules running today**

| What it looks for | Alerts raised | Share of the queue |
| ----------------- | ------------- | ------------------ |
| A single payment of PHP 500,000 or more | 1,540 | 59% |
| An unusual number of payments in a short time | 462 | 18% |
| Amounts sitting just under PHP 500,000 | 366 | 14% |
| A long-inactive account suddenly becoming busy | 126 | 5% |
| Money arriving and leaving again quickly | 124 | 5% |

The first rule is different from the others. **It is not looking for anything suspicious.** Philippine law requires the bank to report any single payment of PHP 500,000 or more, whether or not anything about it seems wrong. The rule exists to meet that obligation. It cannot be switched off — only reordered.

That distinction matters, because it produces nearly six in ten of all alerts. Meanwhile the two rules that look for genuinely unusual **behaviour** — sleeping accounts waking up, and money passing straight through — together produce fewer than one in ten.

## **Her five concerns, and what we can honestly say about each**

| What worries her | Where we address it | What we can say | What we cannot |
| ---------------- | ------------------- | --------------- | -------------- |
| Well-off customers set off alerts again and again, and nothing is ever found | Section 1 | How often big-payment alerts end with nothing found, and how much of the workload they are | That those customers are fine. "Nothing found" is one person's judgement, not a fact |
| Someone splitting a big payment into smaller ones to stay under the limit | Section 2 | How many payments sit just under PHP 500,000, and which customers do it repeatedly | Why they did it. An ordinary payment can look identical |
| Dormant accounts being used to pass money along | Section 3 | What the existing alerts for this found | A before-and-after picture. The data never records when an account woke up |
| Spending that doesn't match what a customer said they earn | Section 4 | How much people moved compared with their stated income, within each job type | That a gap means anything is wrong. Nobody ever checked those income figures |
| The rules may be pointing at the wrong things | Section 5 | How often each rule leads to a real outcome | Anything about the 995 alerts nobody opened |

## **The honest limit on all of it**

There is no field in this data that says who is moving money for someone else. There is no way to work it out either.

The closest thing we have to an answer is what investigators concluded — and that is not the truth. It is what people decided, about only 62% of the alerts, under time pressure, using the very rules we have been asked to judge.

**That is the real constraint this kind of work operates under.** The next section checks what the five sets of records actually contain, and how far they can be trusted, before we ask any of these questions of them.

**Alert dataset (from the team's preparation section).** One row per alert, with its investigation outcome.

In [ ]:
query = """

-- =====================================================
-- STEP 1:
-- SUMMARIZE INVESTIGATIONS TO ONE ROW PER ALERT
-- =====================================================

WITH investigation_summary AS (

    SELECT

        -- Alert being investigated
        alert_id,


        -- Number of investigation records
        COUNT(*) AS investigation_count,


        -- Count SAR/STR-type outcomes
        COUNTIF(
            investigation_result = 'SAR_Filed'
        ) AS sar_count,


        -- Count internally escalated outcomes
        COUNTIF(
            investigation_result = 'Escalated'
        ) AS escalated_count,


        -- Count alerts closed without further action
        COUNTIF(
            investigation_result = 'No_Action'
        ) AS no_action_count,


        -- Count open/pending investigations
        COUNTIF(
            investigation_result = 'Pending'
        ) AS pending_count


    FROM `your-gcp-project.aml_dataset.investigations`


    -- Collapse all case records belonging to the same alert
    GROUP BY alert_id
)


-- =====================================================
-- STEP 2:
-- CREATE ONE ROW PER ALERT
-- =====================================================

SELECT

    -- =================================================
    -- ALERT INFORMATION
    -- =================================================

    al.alert_id,

    al.rule_name,

    al.alert_score,

    al.alert_datetime,


    -- =================================================
    -- TRANSACTION INFORMATION
    -- =================================================

    t.tx_id,

    t.amount,

    t.tx_datetime,

    t.channel,

    t.tx_type,

    t.orig_acct,

    t.dest_acct,


    -- =================================================
    -- ACCOUNT INFORMATION
    -- =================================================

    ac.acct_id,

    ac.cust_id,

    ac.acct_type,

    ac.acct_status,

    ac.open_date,


    -- =================================================
    -- CUSTOMER / KYC INFORMATION
    -- =================================================

    c.age,

    c.occupation,

    c.declared_income,

    c.kyc_risk_score,


    -- =================================================
    -- TRANSACTION FLAGS
    -- =================================================

    CASE
        WHEN t.amount >= 500000
            THEN 1
        ELSE 0
    END AS large_amount_flag,


    CASE
        WHEN t.amount >= 450000
             AND t.amount < 500000
            THEN 1
        ELSE 0
    END AS near_threshold_flag,


    -- =================================================
    -- INVESTIGATION COVERAGE
    -- =================================================

    CASE

        -- If an investigation summary exists,
        -- this alert was investigated
        WHEN inv.alert_id IS NOT NULL
            THEN 1

        ELSE 0

    END AS investigated_flag,


    -- Investigation outcome counts
    COALESCE(inv.investigation_count, 0)
        AS investigation_count,

    COALESCE(inv.sar_count, 0)
        AS sar_count,

    COALESCE(inv.escalated_count, 0)
        AS escalated_count,

    COALESCE(inv.no_action_count, 0)
        AS no_action_count,

    COALESCE(inv.pending_count, 0)
        AS pending_count


FROM `your-gcp-project.aml_dataset.alerts` AS al


-- Connect each alert to its transaction
LEFT JOIN `your-gcp-project.aml_dataset.transactions` AS t

    ON al.tx_id = t.tx_id


-- Connect transaction sender to account information
LEFT JOIN `your-gcp-project.aml_dataset.accounts` AS ac

    ON t.orig_acct = ac.acct_id


-- Attach customer/KYC information
LEFT JOIN `your-gcp-project.aml_dataset.customers` AS c

    ON ac.cust_id = c.cust_id


-- Attach summarized investigation results
LEFT JOIN investigation_summary AS inv

    ON al.alert_id = inv.alert_id

"""

alerts_df = client.query(query).to_dataframe()

print(alerts_df.shape)

alerts_df.head()

# **INVESTIGATION 4 — DOES BEHAVIOUR MATCH THE DECLARED PROFILE?**

**The question.** When a customer opens an account, the bank asks what they do for a living and how much they earn. This section asks a simple follow-up: does the amount of money they actually moved during 2025 look anything like what they said they earn?

**Why it matters.** A person moving far more money than they earn is not doing anything wrong by itself. But it is one of the patterns the bank is supposed to notice, because an account used to pass along somebody else's money looks exactly like this from the outside.

**What we measure.** For every customer we add up all the money they sent out during 2025, then express it as a percentage of the income they declared when they joined the bank. A customer who declared PHP 500,000 and sent out PHP 50,000 comes out at 10%.

**One row per customer.** The query below groups everything to customer level, so each customer appears exactly once and no amount is counted twice.

**What we cannot do.** Declared income was written down by the customer at onboarding. Nobody verified it and it may be years old. So a high percentage may mean unusual activity, or it may just mean somebody understated their income years ago. Nothing in this section identifies a money mule — it produces a list of accounts worth a human look.

## **1. Money moved by each customer**

In [ ]:
query = """

SELECT

    -- =================================================
    -- CUSTOMER PROFILE
    -- =================================================

    c.cust_id,

    c.age,

    c.occupation,

    c.declared_income,

    c.kyc_risk_score,


    -- =================================================
    -- ACCOUNT INFORMATION
    -- =================================================

    -- Number of accounts owned
    COUNT(DISTINCT a.acct_id)
        AS account_count,


    -- =================================================
    -- TRANSACTION ACTIVITY
    -- =================================================

    -- Number of unique outgoing transactions
    COUNT(DISTINCT t.tx_id)
        AS outgoing_transactions,


    -- Total outgoing value during the dataset period
    ROUND(
        SUM(t.amount),
        2
    ) AS outgoing_transaction_value,


    -- Average outgoing transaction amount
    ROUND(
        AVG(t.amount),
        2
    ) AS average_transaction_amount,


    -- Largest outgoing transaction
    ROUND(
        MAX(t.amount),
        2
    ) AS largest_transaction,


    -- =================================================
    -- PROFILE-BEHAVIOUR RATIO
    -- =================================================

    SAFE_DIVIDE(

        -- Total outgoing activity
        SUM(t.amount),

        -- NULLIF changes zero income into NULL,
        -- preventing divide-by-zero errors
        NULLIF(c.declared_income, 0)

    ) AS transaction_income_ratio


FROM `your-gcp-project.aml_dataset.customers` AS c


-- Bring in customer accounts
LEFT JOIN `your-gcp-project.aml_dataset.accounts` AS a

    ON c.cust_id = a.cust_id


-- Bring in transactions sent from those accounts
LEFT JOIN `your-gcp-project.aml_dataset.transactions` AS t

    ON a.acct_id = t.orig_acct


-- We are creating ONE ROW PER CUSTOMER,
-- therefore customer-level fields are our GROUP BY columns
GROUP BY
    c.cust_id,
    c.age,
    c.occupation,
    c.declared_income,
    c.kyc_risk_score


-- Largest profile-behaviour differences first
ORDER BY transaction_income_ratio DESC

"""

# Run the query. One row per customer.
customer_profile_df = client.query(query).to_dataframe()

# Check the grain before anything else: rows should equal customers
print("Rows:", customer_profile_df.shape[0])
print("Different customers:", customer_profile_df["cust_id"].nunique())

customer_profile_df.head()

## **2. Tidying the job titles before we compare**

The occupation field was typed in by hand, so the same job appears written several different ways - "Professional", "professional", "PROFESSIONAL", and one with a space in front of it. To a computer those are four different jobs. If we grouped customers without fixing this first, "Professional" would be split four ways and every group average would be built from a quarter of the people it should include.

The cell below maps every spelling onto the correct one and then checks the result: 18 versions in, 5 real jobs out.

In [ ]:
# The same job written several ways. Left side is what we found in the
# data, right side is what it should be.
occupation_mapping = {
    "professional": "Professional",
    "PROFESSIONAL": "Professional",
    " Professional": "Professional",
    "Professional ": "Professional",
    "business owner": "Business Owner",
    "BUSINESS OWNER": "Business Owner",
    "Business owner": "Business Owner",
    "student": "Student",
    "STUDENT": "Student",
    " Student": "Student",
    "unemployed": "Unemployed",
    "UNEMPLOYED": "Unemployed",
    "Not Employed": "Unemployed"
}

# Swap every wrong spelling for the right one
profile_clean = customer_profile_df.replace({"occupation": occupation_mapping})

# Check it worked. Expect 18 before, 5 after.
print("Job titles before tidying:", customer_profile_df["occupation"].nunique())
print("Job titles after tidying: ", profile_clean["occupation"].nunique())

# Turn the ratio into a percentage so the numbers are readable.
# 0.077 becomes 7.7, which reads as "moved 7.7% of declared income".
profile_clean["pct_of_income_moved"] = round(
    profile_clean["transaction_income_ratio"] * 100,
    1
)

# Some customers cannot be included and we say who, rather than
# letting them disappear quietly.
print("Customers with no job title recorded:",
      profile_clean["occupation"].isnull().sum())
print("Customers with no income figure, so no percentage:",
      profile_clean["pct_of_income_moved"].isnull().sum())

# Keep only customers we can actually place in a group
profile_clean = profile_clean.dropna(
    subset=["occupation", "pct_of_income_moved"]
)

print("Customers included in the comparison below:", profile_clean.shape[0])

## **3. The comparison, by job title**

We do not rank all 10,000 customers against each other, and that decision is the whole point of this section.

If you sort every customer by money-moved-against-income and look at the top, you get a list of business owners - because a business owner moving several times their declared **personal** income is running a business. That is a list, not a finding.

So we compare people against others who declared the same kind of work. A student moving a lot compared with other students is a different signal from a business owner doing the same.

In [ ]:
# Middle value and group size for each job title. The middle value
# (median) is used instead of the average because a handful of customers
# moved hundreds of times their declared income and would drag any
# average somewhere no real customer sits.
occupation_summary = profile_clean.groupby("occupation").agg(
    customers=("cust_id", "nunique"),
    typical_pct=("pct_of_income_moved", "median")
).reset_index()

# Biggest first
occupation_summary = occupation_summary.sort_values(
    "typical_pct",
    ascending=False
)

print(occupation_summary)

# Put the number and the group size into the label so the chart
# is readable on its own
occupation_summary["job_label"] = occupation_summary.apply(
    lambda row: row["occupation"]
                + "\n"
                + str(row["typical_pct"])
                + "%  ("
                + str(row["customers"])
                + " customers)",
    axis=1
)

# Set figure size
plt.figure(figsize=(11, 6), dpi=150)

# Run bar plot
plt.bar(
    occupation_summary["job_label"],
    occupation_summary["typical_pct"],
    color="steelblue"
)

# Set title
plt.title(
    "A typical student moved 7.7% of their declared income in 2025.\n"
    "A typical business owner moved 0.4%.",
    fontsize=14
)

# Set labels
plt.xlabel("Job declared when the account was opened")
plt.ylabel("Money moved as a percentage of declared income")

plt.tight_layout()
plt.show()

In [ ]:
# The bar chart above shows the typical customer in each group.
# This shows the spread - how much customers within one group differ.
plt.figure(figsize=(11, 6), dpi=150)

sns.boxplot(
    x="occupation",
    y="pct_of_income_moved",
    data=profile_clean
)

# Set title
plt.title(
    "Students and unemployed customers are not just higher on average -\n"
    "the whole group sits higher",
    fontsize=13
)

# Set labels
plt.xlabel("Job declared when the account was opened")
plt.ylabel("Money moved as a percentage of declared income")

# A few customers moved hundreds of times their declared income, which
# would squash every box flat. We cut the view at 50% so the boxes are
# readable, and we say below exactly how many customers that hides.
plt.ylim(0, 50)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Never hide customers silently - count them
print("Customers above the 50% cut-off, not visible on the chart:",
      profile_clean[profile_clean["pct_of_income_moved"] > 50].shape[0])

## **4. Is the bank already catching these customers?**

This is the question that decides whether any of the above is useful.

A signal only earns its place if it finds something the current rules miss. If everyone it flags is already in the alert queue, adding it as a new rule would create work without creating coverage.

So the cell below takes every customer who moved **more than their whole declared income** during 2025 and checks a single thing: did they ever raise an alert?

In [ ]:
# Mark every customer who moved more than a full year of declared income
def over_declared_income(row):
    if row["pct_of_income_moved"] > 100:
        return "Moved more than declared income"
    else:
        return "Moved less than declared income"

profile_clean["income_check"] = profile_clean.apply(
    over_declared_income,
    axis=1
)

# Which customers ever raised an alert? alerts_df already carries cust_id,
# so no new query is needed.
customers_with_alerts = alerts_df[["cust_id"]].drop_duplicates(
    subset="cust_id"
)
customers_with_alerts["raised_an_alert"] = "Yes"

# LEFT keeps customers who never raised an alert. An ordinary join would
# throw them away, and they are the comparison group.
profile_checked = profile_clean.merge(
    customers_with_alerts,
    how="left",
    on="cust_id"
)

# A customer with no match never raised an alert
def alert_check(row):
    if row["raised_an_alert"] == "Yes":
        return "Already raised an alert"
    else:
        return "Never raised an alert"

profile_checked["alert_check"] = profile_checked.apply(
    alert_check,
    axis=1
)

# Count the four combinations
overlap = profile_checked.groupby(
    ["income_check", "alert_check"]
).agg(
    customers=("cust_id", "nunique")
).reset_index()

print(overlap)

# The group that matters: moved more than they earn, never flagged
missed_customers = profile_checked[
    (profile_checked["income_check"] == "Moved more than declared income")
    & (profile_checked["alert_check"] == "Never raised an alert")
]

print("Customers moving more than declared income:",
      profile_checked[
          profile_checked["income_check"] == "Moved more than declared income"
      ].shape[0])
print("Of those, never flagged by any rule:", missed_customers.shape[0])
print("Biggest single payment any of them made: PHP",
      missed_customers["largest_transaction"].max())
print("How many of them ever made a single payment of PHP 500,000 or more:",
      missed_customers[
          missed_customers["largest_transaction"] >= 500000
      ].shape[0])

# The list a reviewer would actually work from
review_list = missed_customers.sort_values(
    "pct_of_income_moved",
    ascending=False
)

review_list = review_list[
    [
        "cust_id",
        "occupation",
        "declared_income",
        "outgoing_transactions",
        "outgoing_transaction_value",
        "largest_transaction",
        "pct_of_income_moved"
    ]
]

review_list.head(20)

## **What we found**

**The gap is real, and it is not where you would expect.**

A typical student moved 7.7% of their declared income during 2025. A typical unemployed customer moved 6.0%. A typical professional moved 0.8%, and a typical business owner 0.4%.

So the customers who declared the least earning capacity are the ones whose activity sits furthest from their stated profile - roughly ten times further than professionals. That is the opposite of what most people guess. It matters because recruiting students and people out of work to move money on somebody else's behalf is a well-documented pattern, and this is what that pattern looks like in transaction data.

It is not evidence that anything wrong happened. A student receiving money from family and passing it on produces exactly this shape too.

**Most of these customers are already in the queue.**

469 customers moved more money than their entire declared income. Around three quarters of them had already raised at least one alert - against a rate of 4% across all 10,000 customers. So this measure is picking up something real, but it is mostly finding people the bank has already found.

That has a direct consequence: **this should not become a sixth alert rule.** It would add investigator workload without adding much coverage.

**But a small group slipped through, and the reason is specific.**

Around 120 of those customers raised no alert at all. Not one of them ever made a single payment of PHP 500,000 or more - the largest single payment across the entire group is PHP 479,111.

That is why the rules never saw them. The largest rule looks at one payment at a time and asks whether it crosses PHP 500,000. These customers moved more than a year's declared income across several payments that each stayed underneath. No single payment ever broke the line, so nothing fired.

These customers are higher-priority candidates for review. They are not evidence of wrongdoing.

**What this cannot tell you**

Declared income was written down by the customer when they joined and was never checked. Somebody who understated their income years ago appears in this analysis through no behaviour of their own at all.

The money-moved figure includes transfers a customer made between their own accounts, which raises the total without any money leaving them.

The data dictionary does not say what period declared income covers, so the percentage is only meaningful when comparing one job group against another - not as a number on its own.

305 customers have no income figure recorded (210 blank, 95 entered as zero) and 380 have no job title. All are left out of these charts.

The box plot is cut off at 50% so the boxes can be seen. 542 customers sit above that line - 214 professionals, 175 business owners, 80 students, 69 unemployed and 4 self-employed. Note that the cut-off hides more professionals and business owners than students, which works against the finding rather than for it.

## **What we suggest**

Do not build a new alert rule from this. Three quarters of what it finds is already in the queue.

Instead, run it as a review list once a quarter. It is roughly 120 customers, none of whom the current rules can surface, and reviewing them costs nothing from the alert queue because they are not in it.